# Grafici su debito pubblico

Questa categoria genera o registra grafici su debito, saldi fiscali, entrate, spese e consolidamenti. Per molte serie fiscali coesistono World Bank, IMF, OCPI e ISTAT.

Ogni grafico riporta la fonte primaria usata nella generazione e la dicitura `elaborazione Nazareno Lecis`.


## Come vengono generati

Il notebook separa tre cose: la specifica del grafico, la fonte dati e la trasformazione. Ogni specifica dichiara `titolo`, `tipo_grafico`, `anno_inizio`, `ultimo_dato`, fonte primaria, fonti alternative e cosa viene mostrato. `definisci_grafico_da_indicatore_world_bank` usa direttamente un indicatore WDI; `definisci_grafico_da_rapporto_world_bank` combina due serie WDI; `registra_grafico_da_collegare_a_api` mantiene in inventario un grafico per cui la fonte pubblica esiste ma non e ancora mappata nel codice.

Quando il grafico viene generato, `genera_grafici_e_inventario` scarica i dati via API, applica la trasformazione indicata, costruisce il confronto con Italia, min-max, percentili 25-75 e mediana dei paesi avanzati quando disponibile, poi mostra l'inventario e lascia il plot nel notebook.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "analisi").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from analisi.utils.grafici import (
    PAESI_MONDO,
    definisci_grafico_da_rapporto_world_bank,
    definisci_grafico_da_indicatore_world_bank,
    genera_grafici_e_inventario,
    registra_grafico_da_collegare_a_api,
)

from sources.api_catalog import FONTI_API


## Fonti disponibili per questa categoria

La tabella sotto elenca le API ufficiali considerate per la categoria. Una stessa variabile puo comparire in piu fonti: il notebook indica una fonte primaria per la generazione, ma mantiene le alternative quando esistono definizioni ufficiali equivalenti o vicine.


In [ ]:
fonti_categoria = ('IMF DataMapper API', 'MEF-RGS OpenBDAP API', 'World Bank API', 'ISTAT SDMX API', 'Commissione europea - AMECO API')
catalogo_fonti_api = pd.DataFrame(
    [
        {
            "fonte": fonte.nome,
            "ente": fonte.ente,
            "aree": ", ".join(fonte.aree),
            "note": fonte.note,
            "endpoint_controllo": fonte.url_controllo,
        }
        for fonte in FONTI_API
        if fonte.nome in fonti_categoria
    ]
)
catalogo_fonti_api


## Specifiche dei grafici

Le righe sotto sono volutamente esplicite: per ogni grafico si legge il titolo dell'analisi, il tipo di grafico, la fonte primaria, le fonti ufficiali alternative, l'anno di partenza, l'ultimo dato richiesto e la trasformazione applicata.


In [ ]:
SPECIFICHE_GRAFICI = [
    definisci_grafico_da_indicatore_world_bank(
        titolo='Debito pubblico lordo / PIL',
        nome_output='debito_pubblico_lordo_pil',
        indicatore='GC.DOD.TOTL.GD.ZS',
        trasformazione='level',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Debito pubblico lordo / PIL: livello della serie, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('IMF DataMapper API', 'IMF DataMapper API', 'MEF-RGS OpenBDAP API', 'ISTAT SDMX API'),
    ),
    registra_grafico_da_collegare_a_api(
        titolo='Spese ed entrate totali / PIL',
        nome_output='spese_entrate_totali_pil',
        fonte_primaria='IMF DataMapper API',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        cosa_mostra='Spese ed entrate totali / PIL: Grafico a piu serie: entrate e spese sul PIL.',
        fonti_alternative=('IMF DataMapper API', 'MEF-RGS OpenBDAP API', 'ISTAT SDMX API'),
        note='Grafico a piu serie: entrate e spese sul PIL.',
    ),
    registra_grafico_da_collegare_a_api(
        titolo='Saldo fiscale / PIL',
        nome_output='saldo_fiscale_pil',
        fonte_primaria='IMF DataMapper API',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        cosa_mostra='Saldo fiscale / PIL: Saldo fiscale annuale e cumulato.',
        fonti_alternative=('IMF DataMapper API', 'MEF-RGS OpenBDAP API', 'ISTAT SDMX API'),
        note='Saldo fiscale annuale e cumulato.',
    ),
    registra_grafico_da_collegare_a_api(
        titolo='Saldo primario / PIL',
        nome_output='saldo_primario_pil',
        fonte_primaria='IMF DataMapper API',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        cosa_mostra='Saldo primario / PIL: Saldo primario annuale e cumulato.',
        fonti_alternative=('IMF DataMapper API', 'MEF-RGS OpenBDAP API', 'ISTAT SDMX API'),
        note='Saldo primario annuale e cumulato.',
    ),
    definisci_grafico_da_indicatore_world_bank(
        titolo='Entrate totali / PIL',
        nome_output='entrate_totali_pil',
        indicatore='GC.REV.XGRT.GD.ZS',
        trasformazione='level',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Entrate totali / PIL: livello della serie, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('IMF DataMapper API', 'IMF DataMapper API', 'MEF-RGS OpenBDAP API', 'ISTAT SDMX API'),
        note='WDI usa revenue excluding grants; per replica completa collegare IMF FPP/WEO.',
    ),
    registra_grafico_da_collegare_a_api(
        titolo='Spesa primaria / PIL',
        nome_output='spesa_primaria_pil',
        fonte_primaria='IMF DataMapper API',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        cosa_mostra='Spesa primaria / PIL: Spesa al netto degli interessi.',
        fonti_alternative=('IMF DataMapper API', 'MEF-RGS OpenBDAP API', 'ISTAT SDMX API'),
        note='Spesa al netto degli interessi.',
    ),
    registra_grafico_da_collegare_a_api(
        titolo='Spese per interessi / PIL',
        nome_output='spese_interessi_pil',
        fonte_primaria='IMF DataMapper API',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        cosa_mostra='Spese per interessi / PIL: Interessi sul PIL; WDI disponibile solo come quota delle entrate.',
        fonti_alternative=('IMF DataMapper API', 'MEF-RGS OpenBDAP API', 'ISTAT SDMX API'),
        note='Interessi sul PIL; WDI disponibile solo come quota delle entrate.',
    ),
    definisci_grafico_da_indicatore_world_bank(
        titolo='Spese totali / PIL',
        nome_output='spese_totali_pil',
        indicatore='GC.XPN.TOTL.GD.ZS',
        trasformazione='level',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Spese totali / PIL: livello della serie, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('IMF DataMapper API', 'IMF DataMapper API', 'MEF-RGS OpenBDAP API', 'ISTAT SDMX API'),
    ),
    registra_grafico_da_collegare_a_api(
        titolo='Consolidamenti fiscali',
        nome_output='consolidamenti_fiscali',
        fonte_primaria='IMF DataMapper API',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        cosa_mostra='Consolidamenti fiscali: Episodi, correlazioni e debito netto richiedono WEO multiserie.',
        fonti_alternative=('IMF DataMapper API', 'MEF-RGS OpenBDAP API', 'ISTAT SDMX API'),
        note='Episodi, correlazioni e debito netto richiedono WEO multiserie.',
    ),
]

print(f"Specifiche nella categoria: {len(SPECIFICHE_GRAFICI)}")


## Inventario e generazione

La tabella risultante mostra cosa viene prodotto, quale fonte e stata usata, quali alternative sono possibili, da quale anno parte la serie, fino a quale ultimo dato viene richiesto e quali grafici restano da collegare.


In [ ]:
cartella_output = ROOT / "analisi" / "debito_pubblico"
percorsi, inventario = genera_grafici_e_inventario(SPECIFICHE_GRAFICI, cartella_output)

colonne = [
    "titolo",
    "tipo_grafico",
    "cosa_mostra",
    "fonte_primaria",
    "fonti_alternative",
    "anno_inizio",
    "ultimo_dato",
    "trasformazione",
    "confronto",
    "stato",
    "nome_output",
    "note",
    "errore",
]
print(f"PNG generati: {len(percorsi)}")
for percorso in percorsi:
    print(percorso)
inventario[colonne]


## Plot dei grafici generati

I plot visualizzati qui sono quelli creati dalla cella precedente. I PNG locali restano fuori da Git.


In [ ]:
from IPython.display import Image, display

if not percorsi:
    print("Nessun PNG generato: tutti i grafici della categoria sono ancora da collegare o la fonte non ha restituito dati.")
for percorso in percorsi:
    display(Image(filename=str(percorso)))
